# RAPiDock Colab Runner (Google Drive mode)

This notebook mounts Google Drive and reads/writes files directly from your Drive copy of the project.

Expected project location in Drive:
- `/content/drive/MyDrive/ghsr_af3_molecule_screening`

You can change the path in the configuration cell if needed.

**Important:** RAPiDock ships precompiled Linux modules (for example `PeptideBuilder.so`) that are **not compatible with Colab's default Python 3.12** (you may see `undefined symbol: _PyUnicode_Ready`). This notebook installs a **conda Python 3.10** environment and runs `inference.py` via `conda run`.

In [1]:
#@title 1) Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# If Drive is already mounted from a previous cell in this session, you can skip this cell.

In [2]:
#@title 2) Configure project paths on Drive
import os
from pathlib import Path

PROJECT_DIR = Path('/content/drive/MyDrive/ghsr_af3_molecule_screening')  # change if needed
RAPIDOCK_REPO = Path('/content/RAPiDock')
CONDA_ENV = 'rapidock310'
os.environ['CONDA_ENV'] = CONDA_ENV
# So `!` shell cells find conda (Miniforge installs to this prefix in the next cell).
_conda_bin = '/usr/local/miniforge3/bin'
if Path(_conda_bin).exists():
    os.environ['PATH'] = _conda_bin + os.pathsep + os.environ.get('PATH', '')

_data = PROJECT_DIR / 'data'
# Prefer full receptor; pocket-truncated PDB also works (see README).
_receptor_candidates = [_data / 'ghsr_receptor.pdb', _data / 'ghsr_pocket.pdb']
RECEPTOR_PDB = next((p for p in _receptor_candidates if p.is_file()), _receptor_candidates[0])
BATCH_CSV = PROJECT_DIR / 'inputs_rapidock' / 'protein_peptide.csv'
OUTPUT_SINGLE = PROJECT_DIR / 'outputs_rapidock_test'
OUTPUT_BATCH = PROJECT_DIR / 'outputs_rapidock'

# Make paths visible to subsequent `!shell` cells
for _k, _p in {
    'RECEPTOR_PDB': RECEPTOR_PDB,
    'BATCH_CSV': BATCH_CSV,
    'OUTPUT_SINGLE': OUTPUT_SINGLE,
    'OUTPUT_BATCH': OUTPUT_BATCH,
}.items():
    os.environ[_k] = str(_p)

print('PROJECT_DIR:', PROJECT_DIR)
print('CONDA_ENV:', CONDA_ENV)
print('RECEPTOR_PDB:', RECEPTOR_PDB, 'exists:', RECEPTOR_PDB.is_file())
print('BATCH_CSV exists:', BATCH_CSV.exists())
if not RECEPTOR_PDB.is_file():
    print()
    print('Missing receptor on Drive. Copy from your local clone at least:')
    print('  ', _data / 'ghsr_receptor.pdb')
    print('Or run locally: python scripts/prep_receptor.py --pdb-id 7F9Z --chain-id R --skip-pocket')
    if _data.is_dir():
        _pdbs = sorted(_data.glob('*.pdb'))
        print('data/*.pdb on Drive right now:', [x.name for x in _pdbs] or '(none)')
    else:
        print('Folder missing on Drive:', _data)

In [3]:
#@title 3) Install Miniforge + Python 3.10 env (RAPiDock `.so` requires Python ≤3.11)
# Colab's default Python is 3.12+; bundled PeptideBuilder.so links symbols removed in 3.12.

import os
import shutil
import subprocess
import sys
from pathlib import Path

CONDA_ENV = os.environ.get('CONDA_ENV', 'rapidock310')
MAMBA_ROOT = Path('/usr/local/miniforge3')
CONDA = MAMBA_ROOT / 'bin' / 'conda'
_env_prefix = MAMBA_ROOT / 'envs' / CONDA_ENV


def _conda_env_ok(prefix: Path) -> bool:
    return (prefix / 'conda-meta').is_dir() and (prefix / 'bin' / 'python').is_file()


def _conda(*args: str) -> None:
    env = {**os.environ, 'CONDA_ALWAYS_YES': 'true'}
    r = subprocess.run([str(CONDA), *args], capture_output=True, text=True, env=env)
    if r.stdout:
        print(r.stdout, end='')
    if r.stderr:
        print(r.stderr, end='', file=sys.stderr)
    if r.returncode != 0:
        raise subprocess.CalledProcessError(r.returncode, [str(CONDA), *args], r.stdout, r.stderr)


if not CONDA.exists():
    installer = Path('/tmp/Miniforge3.sh')
    url = 'https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh'
    subprocess.check_call(['wget', '-q', '-O', str(installer), url])
    subprocess.check_call(['bash', str(installer), '-b', '-p', str(MAMBA_ROOT)])

# Drop a half-created env (common after a failed Colab run); `conda env list` can miss it.
if _env_prefix.exists() and not _conda_env_ok(_env_prefix):
    print('Removing incomplete conda env prefix:', _env_prefix)
    shutil.rmtree(_env_prefix, ignore_errors=True)

if not _conda_env_ok(_env_prefix):
    # conda-forge only avoids Anaconda repo TOS prompts that break non-interactive Colab.
    _conda(
        'create',
        '-y',
        '-n',
        CONDA_ENV,
        'python=3.10',
        '--override-channels',
        '-c',
        'conda-forge',
        '--strict-channel-priority',
    )

print('conda env ready:', CONDA_ENV)
subprocess.check_call([str(CONDA), 'run', '-n', CONDA_ENV, 'python', '-V'])

# Colab's system libstdc++ is older than conda-forge ICU (MDAnalysis -> sqlite3 import chain).
_env_lib = MAMBA_ROOT / 'envs' / CONDA_ENV / 'lib'
_base_lib = MAMBA_ROOT / 'lib'
_ld_parts = [str(p) for p in (_env_lib, _base_lib) if p.is_dir()]
if _ld_parts:
    _tail = os.environ.get('LD_LIBRARY_PATH', '').strip()
    os.environ['LD_LIBRARY_PATH'] = os.pathsep.join(_ld_parts + ([_tail] if _tail else []))
    print('Prepended conda libs to LD_LIBRARY_PATH (CXXABI / libstdc++ fix for Colab).')

In [4]:
#@title 4) Clone RAPiDock
!git clone https://github.com/huifengzhao/RAPiDock.git
%cd /content/RAPiDock

In [ ]:
#@title 5) Install RAPiDock dependencies inside the conda env (torch+pyg matched)
import os
import subprocess

import torch

CONDA = '/usr/local/miniforge3/bin/conda'
CONDA_ENV = os.environ.get('CONDA_ENV', 'rapidock310')
# conda-forge Python uses PEP 668 (EXTERNALLY-MANAGED); plain pip installs fail without this in Colab.
_PIP_ENV = {**os.environ, 'PIP_BREAK_SYSTEM_PACKAGES': '1'}
_CONDA_YES = {**os.environ, 'CONDA_ALWAYS_YES': 'true'}

# Do not use `pip install -U pip` here — it often errors under PEP 668. Refresh pip tooling via conda-forge.
subprocess.check_call(
    [
        CONDA,
        'install',
        '-y',
        '-n',
        CONDA_ENV,
        '--override-channels',
        '-c',
        'conda-forge',
        'pip',
        'setuptools',
        'wheel',
    ],
    env=_CONDA_YES,
)

subprocess.check_call(
    [
        CONDA,
        'run',
        '-n',
        CONDA_ENV,
        'python',
        '-m',
        'pip',
        'install',
        'pyyaml',
        'pandas',
        'biopython',
        'MDAnalysis>=2.9.0',
        'fair-esm',
        'e3nn',
        'pyrosetta-installer',
    ],
    env=_PIP_ENV,
)

subprocess.check_call(
    [CONDA, 'run', '-n', CONDA_ENV, 'python', '-m', 'pip', 'install', 'rdkit'],
    env=_PIP_ENV,
)

ver = torch.__version__
if '+' in ver:
    torch_base, pt_cuda = ver.split('+', 1)
else:
    torch_base, pt_cuda = ver, 'cpu'

if pt_cuda == 'cpu':
    torch_index = 'https://download.pytorch.org/whl/cpu'
else:
    torch_index = f'https://download.pytorch.org/whl/{pt_cuda}'

print('Notebook torch (for CUDA wheel selection):', ver)
print('Installing conda-env torch from:', torch_index)

subprocess.check_call(
    [
        CONDA,
        'run',
        '-n',
        CONDA_ENV,
        'python',
        '-m',
        'pip',
        'install',
        f'torch=={torch_base}',
        '--index-url',
        torch_index,
    ],
    env=_PIP_ENV,
)

probe = subprocess.check_output(
    [CONDA, 'run', '-n', CONDA_ENV, 'python', '-c', 'import torch; print(torch.__version__)'],
    text=True,
).strip()
if '+' in probe:
    env_base, env_cuda = probe.split('+', 1)
else:
    env_base, env_cuda = probe, 'cpu'
pyg_tag = f'cu{env_cuda.replace("cu", "")}' if env_cuda.startswith('cu') else 'cpu'
pyg_url = f'https://data.pyg.org/whl/torch-{env_base}+{pyg_tag}.html'

subprocess.run(
    [
        CONDA,
        'run',
        '-n',
        CONDA_ENV,
        'python',
        '-m',
        'pip',
        'uninstall',
        '-y',
        'torch-geometric',
        'torch-scatter',
        'torch-sparse',
        'torch-cluster',
        'pyg-lib',
    ],
    check=False,
    env=_PIP_ENV,
)

subprocess.check_call(
    [
        CONDA,
        'run',
        '-n',
        CONDA_ENV,
        'python',
        '-m',
        'pip',
        'install',
        'pyg_lib',
        'torch_scatter',
        'torch_sparse',
        'torch_cluster',
        'torch_geometric',
        '-f',
        pyg_url,
    ],
    env=_PIP_ENV,
)

subprocess.check_call(
    [
        CONDA,
        'run',
        '-n',
        CONDA_ENV,
        'python',
        '-c',
        'import MDAnalysis, rdkit, torch, torch_geometric, torch_scatter, torch_sparse, torch_cluster; '
        'print("torch:", torch.__version__); '
        'print("MDAnalysis:", MDAnalysis.__version__); '
        'print("RDKit:", rdkit.__version__); '
        'print("torch_geometric:", torch_geometric.__version__)',
    ]
)

# RAPiDock REF2015 scoring needs PyRosetta (for ref2015_score.csv output).
_pyro_check = subprocess.run(
    [CONDA, 'run', '-n', CONDA_ENV, 'python', '-c', 'import pyrosetta; print("PyRosetta OK")'],
    capture_output=True,
    text=True,
)
if _pyro_check.returncode != 0:
    print('PyRosetta is not importable in this conda env. Attempting standard pip install...')
    _pip_cmd = [
        CONDA,
        'run',
        '-n',
        CONDA_ENV,
        'python',
        '-m',
        'pip',
        'install',
        '--upgrade',
        'pyrosetta',
        '--find-links',
        'https://west.rosettacommons.org/pyrosetta/quarterly/release',
    ]
    _pip_install = subprocess.run(_pip_cmd, capture_output=True, text=True, env=_PIP_ENV)
    if _pip_install.returncode != 0:
        print(_pip_install.stdout)
        print(_pip_install.stderr)
        raise RuntimeError(
            'PyRosetta pip install failed. Retry this cell and check network access to RosettaCommons download links.'
        )

_pyro_verify = subprocess.run(
    [
        CONDA,
        'run',
        '-n',
        CONDA_ENV,
        'python',
        '-c',
        'import pyrosetta; print("PyRosetta import OK")',
    ],
    capture_output=True,
    text=True,
)
if _pyro_verify.returncode != 0:
    print(_pyro_verify.stdout)
    print(_pyro_verify.stderr)
    raise RuntimeError(
        'PyRosetta install/import check failed in rapidock310. See error above; usually this is '
        'a wheel/platform mismatch or incomplete install in the env.'
    )
print(_pyro_verify.stdout.strip())

In [ ]:
#@title 6) Download checkpoint
!mkdir -p /content/RAPiDock/train_models/CGTensorProductEquivariantModel
!wget -O /content/RAPiDock/train_models/CGTensorProductEquivariantModel/rapidock_local.pt "https://zenodo.org/records/14193621/files/rapidock_local.pt?download=1"
!ls -lh /content/RAPiDock/train_models/CGTensorProductEquivariantModel/rapidock_local.pt

In [ ]:
#@title 7) Single-case smoke test to Drive output
import os
import subprocess
from pathlib import Path

CONDA = '/usr/local/miniforge3/bin/conda'
_run_env = os.environ.copy()
_run_env['PATH'] = str(Path(CONDA).parent) + os.pathsep + _run_env.get('PATH', '')
_run_env['CONDA_EXE'] = CONDA
_run_env.setdefault('CONDA_ENV', os.environ.get('CONDA_ENV', 'rapidock310'))
_forge = Path('/usr/local/miniforge3')
_ld = os.pathsep.join(
    str(p) for p in (_forge / 'envs' / _run_env['CONDA_ENV'] / 'lib', _forge / 'lib') if p.is_dir()
)
if _ld:
    _run_env['LD_LIBRARY_PATH'] = _ld + os.pathsep + _run_env.get('LD_LIBRARY_PATH', '')

project_dir = Path(os.environ.get('PROJECT_DIR', '/content/drive/MyDrive/ghsr_af3_molecule_screening'))
if 'PROJECT_DIR' not in os.environ:
    os.environ['PROJECT_DIR'] = str(project_dir)

receptor_raw = os.environ.get('RECEPTOR_PDB', '').strip()
if receptor_raw:
    receptor = Path(receptor_raw)
else:
    _data = project_dir / 'data'
    _cands = [_data / 'ghsr_receptor.pdb', _data / 'ghsr_pocket.pdb']
    receptor = next((p for p in _cands if p.is_file()), _cands[0])
    os.environ['RECEPTOR_PDB'] = str(receptor)

output_single = os.environ.get('OUTPUT_SINGLE', '').strip() or str(project_dir / 'outputs_rapidock_test')
os.environ['OUTPUT_SINGLE'] = output_single

if not receptor.is_file():
    raise FileNotFoundError(
        f'Receptor PDB not found: {receptor}\n'
        'Run title 2 first, or copy data/ghsr_receptor.pdb (or ghsr_pocket.pdb) into PROJECT_DIR on Drive.'
    )
Path(output_single).mkdir(parents=True, exist_ok=True)
_infer = Path('/content/RAPiDock/inference.py')
if not _infer.is_file():
    raise FileNotFoundError(f'Missing {_infer}; run title 4) Clone RAPiDock.')
_ckpt = Path('/content/RAPiDock/train_models/CGTensorProductEquivariantModel/rapidock_local.pt')
if not _ckpt.is_file():
    raise FileNotFoundError(f'Missing {_ckpt}; run title 6) Download checkpoint.')

cmd = [
    CONDA, 'run', '--no-capture-output', '-n', _run_env['CONDA_ENV'], 'python',
    '/content/RAPiDock/inference.py',
    '--complex_name', 'hexarelin_test',
    '--protein_description', str(receptor),
    '--peptide_description', 'H[DTR]AW[DPN]K',
    '--output_dir', output_single,
    '--model_dir', '/content/RAPiDock/train_models/CGTensorProductEquivariantModel',
    '--ckpt', 'rapidock_local.pt',
    '--scoring_function', 'ref2015',
    '--N', '1', '--batch_size', '1',
    '--no_final_step_noise',
    '--inference_steps', '8', '--actual_steps', '8',
    '--conformation_partial', '1:1:1',
    '--cpu', '4',
]
proc = subprocess.run(cmd, env=_run_env, capture_output=True, text=True)
if proc.stdout:
    print(proc.stdout, end='')
if proc.stderr:
    print('--- stderr ---')
    print(proc.stderr, end='')
if proc.returncode != 0:
    raise subprocess.CalledProcessError(proc.returncode, cmd, output=proc.stdout, stderr=proc.stderr)

In [ ]:
#@title 8) Full batch run to Drive output (optional)
import os
import subprocess
from pathlib import Path

CONDA = '/usr/local/miniforge3/bin/conda'
_run_env = os.environ.copy()
_run_env['PATH'] = str(Path(CONDA).parent) + os.pathsep + _run_env.get('PATH', '')
_run_env['CONDA_EXE'] = CONDA
_run_env.setdefault('CONDA_ENV', os.environ.get('CONDA_ENV', 'rapidock310'))
_forge = Path('/usr/local/miniforge3')
_ld = os.pathsep.join(
    str(p) for p in (_forge / 'envs' / _run_env['CONDA_ENV'] / 'lib', _forge / 'lib') if p.is_dir()
)
if _ld:
    _run_env['LD_LIBRARY_PATH'] = _ld + os.pathsep + _run_env.get('LD_LIBRARY_PATH', '')

project_dir = Path(os.environ.get('PROJECT_DIR', '/content/drive/MyDrive/ghsr_af3_molecule_screening'))
if 'PROJECT_DIR' not in os.environ:
    os.environ['PROJECT_DIR'] = str(project_dir)

batch_raw = os.environ.get('BATCH_CSV', '').strip() or str(project_dir / 'inputs_rapidock' / 'protein_peptide.csv')
os.environ['BATCH_CSV'] = batch_raw
batch_csv = Path(batch_raw)
if not batch_csv.is_file():
    raise FileNotFoundError(
        f'Batch CSV not found: {batch_csv}\n'
        'Run title 2 first, or copy inputs_rapidock/protein_peptide.csv into PROJECT_DIR on Drive.'
    )

output_batch = os.environ.get('OUTPUT_BATCH', '').strip() or str(project_dir / 'outputs_rapidock')
os.environ['OUTPUT_BATCH'] = output_batch
Path(output_batch).mkdir(parents=True, exist_ok=True)
_infer = Path('/content/RAPiDock/inference.py')
if not _infer.is_file():
    raise FileNotFoundError(f'Missing {_infer}; run title 4) Clone RAPiDock.')
_ckpt = Path('/content/RAPiDock/train_models/CGTensorProductEquivariantModel/rapidock_local.pt')
if not _ckpt.is_file():
    raise FileNotFoundError(f'Missing {_ckpt}; run title 6) Download checkpoint.')

_pyro_probe = subprocess.run(
    [CONDA, 'run', '-n', _run_env['CONDA_ENV'], 'python', '-c', 'import pyrosetta'],
    env=_run_env,
    capture_output=True,
    text=True,
)
if _pyro_probe.returncode != 0:
    raise RuntimeError(
        'PyRosetta not available in conda env. Run title 5 and ensure PyRosetta installs; '
        'otherwise RAPiDock will not write ref2015_score.csv.'
    )

cmd = [
    CONDA, 'run', '--no-capture-output', '-n', _run_env['CONDA_ENV'], 'python',
    '/content/RAPiDock/inference.py',
    '--protein_peptide_csv', str(batch_csv),
    '--output_dir', output_batch,
    '--model_dir', '/content/RAPiDock/train_models/CGTensorProductEquivariantModel',
    '--ckpt', 'rapidock_local.pt',
    '--scoring_function', 'ref2015',
    '--N', '10', '--batch_size', '4',
    '--no_final_step_noise',
    '--inference_steps', '16', '--actual_steps', '16',
    '--conformation_partial', '1:1:1',
    '--cpu', '8',
]
proc = subprocess.run(cmd, env=_run_env, capture_output=True, text=True)
if proc.stdout:
    print(proc.stdout, end='')
if proc.stderr:
    print('--- stderr ---')
    print(proc.stderr, end='')
if proc.returncode != 0:
    raise subprocess.CalledProcessError(proc.returncode, cmd, output=proc.stdout, stderr=proc.stderr)

## Optional: AutoDock Vina rescoring (peptides)

This section rescoring RAPiDock peptide poses with AutoDock Vina and writes a CSV to Drive:
- `results/ranking_peptides_vina.csv`

Notes:
- Vina energies are still approximate; use mainly for relative ranking.
- This uses `rank1_ref2015.pdb` when available, otherwise `rank1.pdb`, per peptide folder in `OUTPUT_BATCH`.

In [ ]:
#@title 10) Install Vina + conversion tools
import shutil
import subprocess
import sys

subprocess.check_call(['apt-get', '-qq', 'update'])
subprocess.check_call(['apt-get', '-qq', 'install', '-y', 'autodock-vina', 'openbabel'])
# PyRosetta/Rosetta rank PDBs often make `obabel` CLI report "0 molecules"; Python bindings still load atoms.
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'openbabel-wheel'])

vina_bin = shutil.which('vina') or shutil.which('vina_1.2')
obabel_bin = shutil.which('obabel')
print('vina:', vina_bin)
print('obabel:', obabel_bin)
if not vina_bin or not obabel_bin:
    raise RuntimeError('Missing vina/openbabel after install.')

In [ ]:
#@title 11) Run Vina rescoring for RAPiDock outputs
import csv
import os
import re
import shutil
import subprocess
import sys
from pathlib import Path

project_dir = Path(os.environ.get('PROJECT_DIR', '/content/drive/MyDrive/ghsr_af3_molecule_screening'))
output_batch = Path(os.environ.get('OUTPUT_BATCH', str(project_dir / 'outputs_rapidock')))
receptor_pdb = Path(os.environ.get('RECEPTOR_PDB', str(project_dir / 'data' / 'ghsr_receptor.pdb')))

if not output_batch.exists():
    raise FileNotFoundError(f'Missing OUTPUT_BATCH: {output_batch}')
if not receptor_pdb.exists():
    raise FileNotFoundError(f'Missing receptor pdb: {receptor_pdb}')

vina_bin = shutil.which('vina') or shutil.which('vina_1.2')
obabel_bin = shutil.which('obabel')
if not vina_bin or not obabel_bin:
    raise RuntimeError('Missing vina/openbabel. Run title 10 first.')

vina_work = project_dir / 'outputs_vina'
vina_work.mkdir(parents=True, exist_ok=True)
# Open Babel often fails to create files directly on Google Drive (FUSE). Write to /tmp, then mirror.
_rx_raw = Path('/tmp/rapidock_vina_receptor_raw.pdbqt')
_rx_proc = subprocess.run(
    [obabel_bin, '-ipdb', str(receptor_pdb), '-opdbqt', '-O', str(_rx_raw)],
    capture_output=True,
    text=True,
)
if _rx_proc.returncode != 0:
    raise RuntimeError(
        f'obabel receptor PDB→PDBQT failed (exit {_rx_proc.returncode}): '
        f'{(_rx_proc.stderr or _rx_proc.stdout or "").strip() or "(no output)"}'
    )
if not _rx_raw.is_file() or _rx_raw.stat().st_size == 0:
    raise RuntimeError(
        f'obabel finished but {_rx_raw} is missing or empty. stderr={_rx_proc.stderr!r}'
    )

# Open Babel PDBQT may include ligand-style ROOT/BRANCH/TORSDOF; Vina rigid receptor rejects those.
_rx_lines = _rx_raw.read_text(encoding='utf-8', errors='ignore').splitlines()
_rx_out = []
for _ln in _rx_lines:
    _s = _ln.strip()
    if not _s:
        continue
    _tok = _s.split(None, 1)[0]
    if _tok in {'ROOT', 'ENDROOT', 'TORSDOF', 'BEGIN_RES', 'END_RES', 'UNBOUND'}:
        continue
    if _tok.startswith(('BRANCH', 'ENDBRANCH')):
        continue
    _rx_out.append(_ln.rstrip())
_receptor_clean = Path('/tmp/rapidock_vina_receptor.pdbqt')
_receptor_clean.write_text('\n'.join(_rx_out) + '\n', encoding='utf-8')
receptor_pdbqt = _receptor_clean
_rx_drive = vina_work / 'receptor.pdbqt'
try:
    shutil.copy2(_receptor_clean, _rx_drive)
except OSError as _e:
    print(f'Note: could not copy receptor PDBQT to Drive ({_rx_drive}): {_e}')
else:
    print('Receptor PDBQT (mirror on Drive):', _rx_drive)

# Auto-derive a docking box from receptor coordinates with a small padding.
mins = [float('inf')] * 3
maxs = [float('-inf')] * 3
with receptor_pdb.open(encoding='utf-8', errors='ignore') as fh:
    for line in fh:
        if line.startswith(('ATOM', 'HETATM')) and len(line) >= 54:
            x = float(line[30:38])
            y = float(line[38:46])
            z = float(line[46:54])
            mins[0] = min(mins[0], x)
            mins[1] = min(mins[1], y)
            mins[2] = min(mins[2], z)
            maxs[0] = max(maxs[0], x)
            maxs[1] = max(maxs[1], y)
            maxs[2] = max(maxs[2], z)

if not all(m < 1e20 for m in mins):
    raise RuntimeError('Failed to parse receptor coordinates for box setup.')

padding = 4.0
center = [(a + b) / 2.0 for a, b in zip(mins, maxs)]
size = [max(18.0, (b - a) + padding) for a, b in zip(mins, maxs)]

rows = []


def _ligand_pdb_to_pdbqt_py(pdb_path: Path, pdbqt_path: Path) -> None:
    """Convert ligand PDB → PDBQT using Open Babel's Python API.

    PyRosetta `rank1_ref2015.pdb` files are often rejected by the `obabel` CLI
    (stderr: "not a valid PDB file" / "0 molecules converted") while the library
    still loads coordinates and can write PDBQT.
    """
    try:
        from openbabel import openbabel as ob
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'openbabel-wheel'])
        from openbabel import openbabel as ob
    conv = ob.OBConversion()
    conv.SetInAndOutFormats('pdb', 'pdbqt')
    mol = ob.OBMol()
    conv.ReadFile(mol, str(pdb_path))
    if mol.NumAtoms() == 0:
        raise RuntimeError(
            f'Open Babel loaded 0 atoms from {pdb_path}. '
            'Check for ATOM/HETATM lines in the pose PDB.'
        )
    if not conv.WriteFile(mol, str(pdbqt_path)):
        raise RuntimeError(f'Open Babel failed writing PDBQT to {pdbqt_path}')
    if not pdbqt_path.is_file() or pdbqt_path.stat().st_size == 0:
        raise RuntimeError(f'Open Babel wrote an empty PDBQT file: {pdbqt_path}')


def _ligand_pdbqt_rigid_single_root(pdbqt_path: Path) -> None:
    """Make Open Babel peptide PDBQT compatible with AutoDock Vina's ligand parser.

    Babel often emits multiple fragments, many BRANCH blocks, and multiple TORSDOF lines.
    Vina only reads atoms between the first ROOT/ENDROOT and allows a single TORSDOF line,
    which can yield 'No atoms in this ligand' for peptides. For --score_only, a rigid
    ligand (all atoms in one ROOT, TORSDOF 0) is sufficient.
    """
    lines = pdbqt_path.read_text(encoding='utf-8', errors='ignore').splitlines()
    atoms = []
    for ln in lines:
        if ln.startswith('ATOM ') or ln.startswith('HETATM'):
            if ln.startswith('HETATM'):
                ln = 'ATOM  ' + ln[6:]
            atoms.append(ln)
    if not atoms:
        raise RuntimeError(f'Open Babel produced no ATOM/HETATM lines: {pdbqt_path}')
    out = ['ROOT']
    for i, ln in enumerate(atoms, start=1):
        if len(ln) < 12:
            raise RuntimeError(f'Unexpected short PDBQT line ({len(ln)} chars): {ln!r}')
        out.append(f'ATOM  {i:5d}{ln[11:]}')
    out.append('ENDROOT')
    out.append('TORSDOF 0')
    pdbqt_path.write_text('\n'.join(out) + '\n', encoding='utf-8')


for complex_dir in sorted(output_batch.iterdir()):
    if not complex_dir.is_dir():
        continue
    name = complex_dir.name
    ligand_pdb = complex_dir / 'rank1_ref2015.pdb'
    if not ligand_pdb.exists():
        ligand_pdb = complex_dir / 'rank1.pdb'
    if not ligand_pdb.exists():
        # Keep quiet for unrelated folders; only score folders with a known top-pose filename.
        continue

    out_dir = vina_work / name
    out_dir.mkdir(parents=True, exist_ok=True)
    ligand_pdbqt_drive = out_dir / 'rank1.pdbqt'
    _safe = re.sub(r'[^0-9a-zA-Z_-]+', '_', name).strip('_') or 'lig'
    _lig_tmp = Path(f'/tmp/rapidock_vina_lig_{_safe}.pdbqt')
    try:
        _ligand_pdb_to_pdbqt_py(ligand_pdb, _lig_tmp)
    except RuntimeError as _e:
        raise RuntimeError(f'{name}: {_e}') from _e

    _ligand_pdbqt_rigid_single_root(_lig_tmp)

    try:
        shutil.copy2(_lig_tmp, ligand_pdbqt_drive)
    except OSError as _e:
        print(f'Note: could not copy ligand PDBQT to Drive ({ligand_pdbqt_drive}): {_e}')

    ligand_pdbqt = _lig_tmp

    cmd = [
        vina_bin,
        '--receptor', str(receptor_pdbqt),
        '--ligand', str(ligand_pdbqt),
        '--center_x', f'{center[0]:.3f}',
        '--center_y', f'{center[1]:.3f}',
        '--center_z', f'{center[2]:.3f}',
        '--size_x', f'{size[0]:.3f}',
        '--size_y', f'{size[1]:.3f}',
        '--size_z', f'{size[2]:.3f}',
        '--score_only',
    ]
    proc = subprocess.run(cmd, capture_output=True, text=True)
    text = (proc.stdout or '') + '\n' + (proc.stderr or '')
    (out_dir / 'vina_score_only.log').write_text(text, encoding='utf-8')

    if proc.returncode != 0:
        print(f'Vina failed for {name}; see {out_dir / "vina_score_only.log"}')
        rows.append({'name': name, 'vina_best_kcalmol': None, 'vina_source_pose': str(ligand_pdb)})
        continue

    score = None
    patterns = [
        r'Estimated Free Energy of Binding\s*:\s*([-+]?\d+(?:\.\d+)?)',
        r'Affinity:\s*([-+]?\d+(?:\.\d+)?)\s*\(kcal/mol\)',
    ]
    for pat in patterns:
        m = re.search(pat, text)
        if m:
            score = float(m.group(1))
            break

    if score is None:
        print(f'Could not parse Vina score for {name}; see {out_dir / "vina_score_only.log"}')

    rows.append({'name': name, 'vina_best_kcalmol': score, 'vina_source_pose': str(ligand_pdb)})

finite = [r['vina_best_kcalmol'] for r in rows if isinstance(r['vina_best_kcalmol'], float)]
best = min(finite) if finite else None
for r in rows:
    v = r['vina_best_kcalmol']
    r['vina_delta_from_best_kcalmol'] = (v - best) if (best is not None and isinstance(v, float)) else None

rows.sort(key=lambda r: (float('inf') if r['vina_best_kcalmol'] is None else r['vina_best_kcalmol']))
for i, r in enumerate(rows, start=1):
    r['vina_rank'] = i

out_csv = project_dir / 'results' / 'ranking_peptides_vina.csv'
out_csv.parent.mkdir(parents=True, exist_ok=True)
with out_csv.open('w', newline='', encoding='utf-8') as fh:
    writer = csv.DictWriter(
        fh,
        fieldnames=['name', 'vina_best_kcalmol', 'vina_delta_from_best_kcalmol', 'vina_rank', 'vina_source_pose'],
    )
    writer.writeheader()
    for r in rows:
        writer.writerow(r)

print('Wrote:', out_csv)
print('Peptides scored:', len(rows))
print('Box center:', center)
print('Box size:', size)

In [ ]:
#@title 12) Preview Vina peptide ranking
import pandas as pd
from pathlib import Path
import os

project_dir = Path(os.environ.get('PROJECT_DIR', '/content/drive/MyDrive/ghsr_af3_molecule_screening'))
vina_csv = project_dir / 'results' / 'ranking_peptides_vina.csv'
if not vina_csv.exists():
    raise FileNotFoundError(f'Missing {vina_csv}. Run title 11 first.')

df = pd.read_csv(vina_csv)
display(df.sort_values('vina_rank'))

## Optional: 3D structure preview (receptor + peptide)

RAPiDock writes the receptor and each ranked peptide pose as **separate** PDB files under `OUTPUT_BATCH` / `outputs_rapidock/<complex_name>/`:

- `<complex_name>_protein_raw.pdb` — receptor used for docking (often chain **R**).
- `rank<N>_ref2015.pdb` (or `rank<N>.pdb`) — peptide coordinates only (often chain **A**).

Run **title 13** after batch outputs exist (title 8). Set `COMPLEX_NAME` to a folder name under your batch output directory (for example `ipamorelin`).

For **GHS-R + small molecules** (Boltz), use py3Dmol in [`boltz2_ghsr_screen.ipynb`](boltz2_ghsr_screen.ipynb) and `results/top_poses/*.pdb`; see README section *3D structure visualization*.

In [6]:
#@title 13) Optional: 3D preview (receptor + peptide, py3Dmol)
# Run after batch outputs exist (title 8). Uses PROJECT_DIR and OUTPUT_BATCH from title 2.
import os
import subprocess
import sys
from pathlib import Path

try:
    import py3Dmol
except ModuleNotFoundError:
    subprocess.check_call(
        [sys.executable, '-m', 'pip', 'install', '-q', 'py3dmol>=2.0.0'],
    )
    import py3Dmol

project_dir = Path(os.environ.get('PROJECT_DIR', '/content/drive/MyDrive/ghsr_af3_molecule_screening'))
output_batch = Path(os.environ.get('OUTPUT_BATCH', str(project_dir / 'outputs_rapidock')))

COMPLEX_NAME = 'ipamorelin'  # @param {type:"string"}
POSE_RANK = 1  # @param {type:"integer"}

complex_dir = output_batch / COMPLEX_NAME
if not complex_dir.is_dir():
    raise FileNotFoundError(
        f'Missing batch folder: {complex_dir}. Run title 2 (paths) and title 8 (batch), or fix COMPLEX_NAME.'
    )

receptor_pdb = complex_dir / f'{COMPLEX_NAME}_protein_raw.pdb'
if not receptor_pdb.is_file():
    _cands = sorted(complex_dir.glob('*_protein_raw.pdb'))
    if _cands:
        receptor_pdb = _cands[0]
    else:
        raise FileNotFoundError(
            f'No *_protein_raw.pdb in {complex_dir}. Expected {COMPLEX_NAME}_protein_raw.pdb'
        )

peptide_pdb = complex_dir / f'rank{POSE_RANK}_ref2015.pdb'
if not peptide_pdb.is_file():
    peptide_pdb = complex_dir / f'rank{POSE_RANK}.pdb'
if not peptide_pdb.is_file():
    raise FileNotFoundError(
        f'Missing pose: rank{POSE_RANK}_ref2015.pdb or rank{POSE_RANK}.pdb under {complex_dir}'
    )

rec = receptor_pdb.read_text(encoding='utf-8', errors='replace')
pep = peptide_pdb.read_text(encoding='utf-8', errors='replace')

view = py3Dmol.view(width=900, height=600)
view.addModel(rec, 'pdb')
view.addModel(pep, 'pdb')
view.setStyle({'model': 0}, {'cartoon': {'color': 'lightgray'}})
view.setStyle({'model': 1}, {'stick': {}, 'cartoon': {'color': 'cyan'}})
view.zoomTo()
view.show()

print('Receptor:', receptor_pdb)
print('Peptide: ', peptide_pdb)

In [ ]:
#@title 9) Verify Drive outputs
import os
import subprocess
from pathlib import Path


def _show_tree(label: str, raw: str) -> None:
    raw = (raw or '').strip()
    print(f'{label}: {raw or "(not set)"}')
    if not raw:
        print('  (not set — run title 2) Configure project paths first.)')
        return
    root = Path(raw)
    if not root.exists():
        print('  (directory does not exist yet — run titles 7 / 8 after inputs exist on Drive.)')
        return
    subprocess.run(['ls', '-R', str(root)], check=False)


_show_tree('OUTPUT_SINGLE', os.environ.get('OUTPUT_SINGLE', ''))
print('---')
_show_tree('OUTPUT_BATCH', os.environ.get('OUTPUT_BATCH', ''))

## Next step (local parsing)

After Colab finishes, go back to your local repo and run:

```bash
python scripts/parse_rapidock_results.py
python scripts/parse_results.py
python scripts/visualize.py
```

Because outputs are written directly to Drive, no manual download/upload step is needed.